# EUR_PC1_10b — retraining ablation PC1, arm 10b (Track F, chat D32-F)

**Design:** `results/eureka_d32f/DESIGN_PC1.md` (approved by the author 20/11/2026 16:17). Arm **10b**: the GAE retrained WITHOUT the gamma powers, relative and absolute (node input 28, target 10).
The thesis training recipe (12 training patients' clean windows of the record data, V5's constants, Adam 1e-3, cosine,
batch 32, **200 epochs, seeds 42 / 1 / 2 / 3**, 4 in parallel), gates **G-pc1const** and **G-pc1vec** first; then the V5
chain on V0 and the arm (k recalibrated), with the usual SMOKE and record gates. **Accelerator:** None (CPU).
**Internet:** ON. **Inputs (Add Input):** `v9-x3-fold1`, `v9-x3-fold3`, `thesis-r2-external-features`. **Output:**
`/kaggle/working/eureka/pc1_10b` (→ `results/eureka_d32f/pc1_10b`; the checkpoints are ≈ 20 KB each). **Runtime:**
≈ 4.5–5 h (computed: 3.9 h measured for 4 parallel trainings in R1_RUN + ≈ 0.4 h scoring). Code: `github.com/Nhanvas/Thesis`
ref `main`; module sha256 asserted below. **Nothing printed here is a research result.** Do not edit;
*Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
NOTEBOOK, REPO_REF = "EUR_PC1_10b", "main"
TEST = bool(os.environ.get("EUR_TEST_REPO"))                     # synthetic test in the build container only
WORK = os.environ.get("EUR_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("EUR_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("EUR_TEST_SCRATCH", "/tmp/r2")
if TEST:
    REPO = os.environ["EUR_TEST_REPO"]; HEAD = "test"
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/eur_abl.py"):
        subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                              f"&& cd {REPO} && git checkout -q {REPO_REF}", shell=True)
    HEAD = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("repository", REPO, HEAD, f"({time.time()-T_NB:.0f}s)")

In [ ]:
# START: sha256 of every module (the reviewed code only), environment, imports
WANT = {"eur_abl.py": "04412808c0088fab", "r2_r1.py": "76cbe51ff0180d4a", "r2_core.py": "902d15ad4feb3e8a", "e16_recording_baseline.py": "f170e1c59f5b0447", "e17b_latent_family.py": "3b4f034dca00eaf3", "p3c_freeze.py": "dabe4139f6005fd2", "p3_levers.py": "feb247d2d08c52ba", "p2_scalars.py": "11feeaf3ba95ced4", "e22_test_oneshot.py": "669350bbfaaf0f4e", "e23_test_look7.py": "88587668baf12a64", "x1_external.py": "9962e3e7fe4ddc35", "p2_dump_seeds23.py": "0bc887542989310f", "e18_cpd_localizer.py": "e18c0ffe37f78e77", "e10_kernel_cpd.py": "5c692e33d92fd0fb", "f1_core.py": "bf4bfd69d7669226", "model_cmp_core.py": "179b91278035b37a", "retrain/gae_joint.py": "06b0566c97cd0ec4", "retrain/train_gae_v2.py": "2503a0e01ac9691e"}
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
bad = {n: sha(f"{REPO}/src/{n}") for n in WANT if sha(f"{REPO}/src/{n}") != WANT[n]}
assert not bad, f"modules differ from the reviewed code (rebuild the notebook or pin the right commit): {bad}"
os.environ.update(EUR_VARIANTS="V0,10b", EUR_TAG="pc1_10b", THESIS_REPO=REPO, R2_WORK=WORK, R2_INPUT=INPUT, R2_SCRATCH=SCRATCH, OMP_NUM_THREADS="1",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
if TEST:
    os.environ["R2_TEST"] = "1"
sys.path[:0] = [REPO + "/src"]
import eur_abl as M
assert M.__file__ == f"{REPO}/src/eur_abl.py", M.__file__
print("CPU cores:", os.cpu_count(), "| eur_abl", M.CODE_SHA, "| variants", list(M.VARIANTS))

In [ ]:
# RUN: G-self, G-data, SMOKE (R1.0 rows, G-D0, G-enc, G-x0, G-eng, projection), then the full run
ERR = None
try:
    M.main(NOTEBOOK, HEAD, workers=int(os.environ.get("EUR_WORKERS", "0")) or None)
except SystemExit as e:
    ERR = str(e)

In [ ]:
# END: gates, projection, outputs, measured runtime
OUT = os.path.join(WORK, "eureka", "pc1_10b")
for f in ("pc1_training.json", "gates.json", "smoke/gates.json", "projection.json", "k.csv", "run_meta.json"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        print("=====", f); print(open(p).read()[:4000])
for root, _, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(root, f); print(os.path.relpath(p, WORK), os.path.getsize(p))
print(f"MEASURED notebook runtime: {(time.time() - T_NB) / 3600:.3f} h")
if ERR:
    raise RuntimeError(f"STOPPED: {ERR}")